# 📡 Day 02a: Broadcasting — Eliminate Loops Forever

---

## 🎯 What You'll Master Today
- Broadcasting rules (the 3 rules)
- Vectorized operations vs loops
- Performance comparison
- 10 broadcasting puzzles

**Goal:** Predict broadcasting output without running code.

---

## 🎭 The Stretch Analogy

Imagine two sheets of different sizes. Broadcasting **stretches** the smaller sheet to match the bigger one — virtually, without copying data.

```
╔═══════════════════════════════════════════════════════════════════╗
║  BROADCASTING — 3 RULES                                          ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Rule 1: If arrays differ in ndim, prepend 1s to smaller shape  ║
║          (3,4) + (4,) → (3,4) + (1,4)                          ║
║                                                                   ║
║  Rule 2: Arrays with size 1 along a dim are stretched           ║
║          (3,4) + (1,4) → (3,4) + (3,4) ← 1 stretched to 3    ║
║                                                                   ║
║  Rule 3: If sizes differ and neither is 1 → ERROR              ║
║          (3,4) + (2,4) → ❌ Can't broadcast 3 and 2            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import numpy as np

# ============================================================
# Basic Broadcasting Examples
# ============================================================

# Scalar + array (scalar broadcast to entire array)
a = np.array([1, 2, 3])
print(f"[1,2,3] + 10 = {a + 10}")  # [11, 12, 13]

# 2D + 1D (row broadcast across all rows)
mat = np.array([[1, 2, 3],
                [4, 5, 6]])
row = np.array([10, 20, 30])
print(f"\n(2,3) + (3,) = \n{mat + row}")
# Step: (3,) → (1,3) → (2,3), then add

# 2D + column vector
col = np.array([[100], [200]])  # Shape (2, 1)
print(f"\n(2,3) + (2,1) = \n{mat + col}")
# (2,1) → (2,3), each row gets its own value

In [ ]:
# ============================================================
# Shape Compatibility Check
# ============================================================

def can_broadcast(shape1, shape2):
    """Check if two shapes are broadcastable."""
    # Pad shorter shape with 1s
    s1 = list(shape1)
    s2 = list(shape2)
    max_len = max(len(s1), len(s2))
    s1 = [1] * (max_len - len(s1)) + s1
    s2 = [1] * (max_len - len(s2)) + s2
    
    result = []
    for d1, d2 in zip(s1, s2):
        if d1 == d2:
            result.append(d1)
        elif d1 == 1:
            result.append(d2)
        elif d2 == 1:
            result.append(d1)
        else:
            return None  # Can't broadcast
    return tuple(result)

pairs = [
    ((3, 4), (4,)),
    ((3, 4), (3, 1)),
    ((3, 4), (2, 4)),
    ((5, 3, 1), (1, 4)),
    ((256, 256, 3), (3,)),
]

for s1, s2 in pairs:
    result = can_broadcast(s1, s2)
    if result:
        print(f"  {s1} + {s2} → {result} ✅")
    else:
        print(f"  {s1} + {s2} → INCOMPATIBLE ❌")

In [ ]:
# ============================================================
# Performance: Loops vs Vectorized
# ============================================================
import time

N = 1_000_000
a = np.random.randn(N)
b = np.random.randn(N)

# Loop version
start = time.perf_counter()
result_loop = np.empty(N)
for i in range(N):
    result_loop[i] = a[i] + b[i]
loop_time = time.perf_counter() - start

# Vectorized version
start = time.perf_counter()
result_vec = a + b
vec_time = time.perf_counter() - start

print(f"Loop:       {loop_time:.4f}s")
print(f"Vectorized: {vec_time:.6f}s")
print(f"Speedup:    {loop_time/vec_time:.0f}x faster!")
print(f"\n💡 Rule: If you're writing a for loop over array elements, you're doing it wrong.")

In [ ]:
# ============================================================
# 10 Broadcasting Puzzles — Predict, Then Verify
# ============================================================

# Puzzle 1: Normalize columns (subtract mean of each column)
data = np.array([[1, 2, 3],
                 [4, 5, 6],
                 [7, 8, 9]])
col_means = data.mean(axis=0)  # Shape: (3,)
normalized = data - col_means   # (3,3) - (3,) → works!
print(f"P1 — Col-normalized means: {normalized.mean(axis=0)}")

# Puzzle 2: Normalize rows (subtract mean of each row)
row_means = data.mean(axis=1, keepdims=True)  # Shape: (3,1) — keepdims!
normalized = data - row_means   # (3,3) - (3,1) → works!
print(f"P2 — Row-normalized means: {normalized.mean(axis=1)}")

# Puzzle 3: Outer product
x = np.array([1, 2, 3])   # (3,)
y = np.array([4, 5])      # (2,)
outer = x[:, None] * y[None, :]  # (3,1) * (1,2) → (3,2)
print(f"P3 — Outer product: \n{outer}")

# Puzzle 4: Pairwise distances
points = np.array([1.0, 3.0, 5.0])  # 3 points
# distances[i,j] = |points[i] - points[j]|
distances = np.abs(points[:, None] - points[None, :])  # (3,1)-(1,3) → (3,3)
print(f"P4 — Pairwise distances: \n{distances}")

In [ ]:
# Puzzle 5: Scale each row by a different factor
mat = np.ones((3, 4))
scales = np.array([1, 2, 3])[:, None]  # (3,1)
print(f"P5 — Scaled rows: \n{mat * scales}")

# Puzzle 6: Add bias to each sample (neural network style)
# batch (32, 10) + bias (10,) → broadcast!
batch = np.random.randn(32, 10)
bias = np.random.randn(10)
result = batch + bias  # (32,10) + (10,) → (32,10)
print(f"P6 — Batch+bias shape: {result.shape}")

# Puzzle 7: Image normalization (H, W, C) / (C,)
image = np.random.randint(0, 256, (224, 224, 3)).astype(float)
channel_means = np.array([123.68, 116.78, 103.94])  # ImageNet means
normalized = image - channel_means  # (224,224,3) - (3,) → works!
print(f"P7 — Normalized image shape: {normalized.shape}")

# Puzzle 8: Softmax (numerically stable)
logits = np.array([[1.0, 2.0, 3.0],
                   [1.0, 5.0, 2.0]])
# Subtract max per row for numerical stability
shifted = logits - logits.max(axis=1, keepdims=True)  # (2,3) - (2,1)
exp = np.exp(shifted)
softmax = exp / exp.sum(axis=1, keepdims=True)  # (2,3) / (2,1)
print(f"P8 — Softmax: \n{softmax}")
print(f"     Row sums: {softmax.sum(axis=1)}")  # Should be [1, 1]

In [ ]:
# Puzzle 9: One-hot encoding without loops
labels = np.array([0, 2, 1, 0, 3])  # 5 samples, 4 classes
n_classes = 4
one_hot = (labels[:, None] == np.arange(n_classes)[None, :]).astype(int)
# (5,1) == (1,4) → (5,4) boolean → int
print(f"P9 — One-hot: \n{one_hot}")

# Puzzle 10: Euclidean distance matrix
X = np.array([[0, 0], [1, 1], [2, 2]])  # 3 points in 2D
# dist[i,j] = ||X[i] - X[j]||
diff = X[:, None, :] - X[None, :, :]  # (3,1,2) - (1,3,2) → (3,3,2)
dist = np.sqrt((diff ** 2).sum(axis=-1))  # (3,3)
print(f"P10 — Distance matrix: \n{np.round(dist, 2)}")

---

## 🗺️ Broadcasting Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  BROADCASTING PATTERNS FOR ML                                     ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  NORMALIZE COLUMNS:  data - data.mean(axis=0)                   ║
║  NORMALIZE ROWS:     data - data.mean(axis=1, keepdims=True)    ║
║  OUTER PRODUCT:      x[:,None] * y[None,:]                      ║
║  PAIRWISE DIST:      |x[:,None] - y[None,:]|                   ║
║  SOFTMAX:            exp(x-max) / sum(exp(x-max), keepdims)    ║
║  ONE-HOT:            labels[:,None] == arange(n)[None,:]        ║
║  BATCH + BIAS:       (N,D) + (D,)   ← neural network          ║
║  IMAGE NORMALIZE:    (H,W,C) - (C,) ← channel means           ║
║                                                                   ║
║  REMEMBER: keepdims=True when broadcasting along axis=1         ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | What is broadcasting? | NumPy's way to perform ops on different-shaped arrays by virtually stretching |
| 2 | Rules of broadcasting? | 1) Prepend 1s to shorter shape 2) Size-1 dims stretch 3) Else error |
| 3 | Why avoid loops? | Vectorized ops use C backend + SIMD, 100-1000x faster than Python loops |
| 4 | What is keepdims? | Keeps reduced dimension as size 1 for broadcasting compatibility |
| 5 | Implement softmax? | exp(x - max(x, keepdims)) / sum(exp(...), keepdims) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • 3 rules: prepend 1s → stretch 1s → else error       │
## │  • keepdims=True is essential for row operations         │
## │  • [:,None] adds column dim, [None,:] adds row dim      │
## │  • Vectorized is 100-1000x faster than loops            │
## │  • Master softmax, normalization, distance patterns     │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Linear Algebra** — np.linalg, PCA from scratch